# DS Sprint Assessment – Telco Customer Churn Prediction
**Student Details**
- **Name:** *(Fill your name)*
- **Register Number:** *(Fill your Reg. No.)*
- **Class:** 2024 CSE
- **Challenge ID:** DS-CHN-01
- **Challenge Title:** Telco Customer Churn Prediction

---

## Round 1 – Data Understanding, Preparation & EDA

### Problem Statement
Telecom companies lose significant revenue due to customer churn.
This project predicts whether a customer will leave the service based on
demographic, account, and usage data, enabling proactive retention strategies.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from xgboost import XGBClassifier
import openpyxl
import os

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
print('All libraries imported successfully.')

All libraries imported successfully.


In [2]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('telco_churn.csv')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print('First 5 rows:')
display(df.head())

Dataset Shape: 7043 rows x 21 columns

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# ── Dataset Info & Stats ─────────────────────────────────────────────────────
print('=== DATASET DETAILS ===')
print(f'Source       : IBM Telco Customer Churn (Kaggle / IBM Sample Data)')
print(f'Records      : {df.shape[0]}')
print(f'Attributes   : {df.shape[1]}')
print()
df.info()

=== DATASET DETAILS ===
Source       : IBM Telco Customer Churn (Kaggle / IBM Sample Data)
Records      : 7043
Attributes   : 21

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies  

In [4]:
display(df.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Data Preprocessing

In [5]:
# ── 1. Fix TotalCharges (blank strings -> NaN -> 0) ───────────────────────────
# The 11 blanks are brand-new customers (tenure = 0) who have not been billed yet,
# so 0 is the correct value (a median would invent a bill).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_before = df['TotalCharges'].isnull().sum()
assert (df.loc[df['TotalCharges'].isnull(), 'tenure'] == 0).all()
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f'Missing TotalCharges before: {missing_before}  |  After fill: {df.isnull().sum().sum()}')

Missing TotalCharges before: 11  |  After fill: 0


In [6]:
# ── 2. Duplicate Removal ──────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Dataset after dedup : {df.shape[0]} rows')

Duplicate rows found: 0
Dataset after dedup : 7043 rows


In [7]:
# ── 3. Outlier Detection (IQR method, report only) ────────────────────────────
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print('Outlier counts (IQR method):')
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'  {col}: {n} outliers')
print('No outliers found, so no clipping was needed.')

Outlier counts (IQR method):
  tenure: 0 outliers
  MonthlyCharges: 0 outliers
  TotalCharges: 0 outliers
No outliers found, so no clipping was needed.


### EDA – Exploratory Data Analysis

In [8]:
# ── EDA Chart 1: Churn Distribution ──────────────────────────────────────────
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#4CAF50', '#F44336']

bars = axes[0].bar(churn_counts.index, churn_counts.values,
                   color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Churn Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Churn'); axes[0].set_ylabel('Count')
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(val), ha='center', va='bottom', fontweight='bold')

axes[1].pie(churn_pct.values, labels=['No Churn','Churn'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0, 0.05), shadow=True)
axes[1].set_title('Churn Percentage', fontsize=13, fontweight='bold')

plt.suptitle('Figure 1: Target Variable – Churn Distribution', fontsize=12)
plt.tight_layout()
plt.savefig('chart_01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'EDA Finding 1: Dataset is imbalanced — {churn_pct["Yes"]:.1f}% churned vs {churn_pct["No"]:.1f}% retained.')

EDA Finding 1: Dataset is imbalanced — 26.5% churned vs 73.5% retained.


In [9]:
# ── EDA Chart 2: Tenure & Monthly Charges vs Churn ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [('No','#4CAF50'), ('Yes','#F44336')]:
    axes[0].hist(df[df['Churn']==label]['tenure'], bins=30,
                 alpha=0.6, color=color, label=f'Churn={label}', edgecolor='white')
axes[0].set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tenure (months)'); axes[0].set_ylabel('Count')
axes[0].legend()

sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={'No':'#4CAF50','Yes':'#F44336'}, ax=axes[1])
axes[1].set_title('Monthly Charges by Churn', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Churn'); axes[1].set_ylabel('Monthly Charges ($)')

plt.suptitle('Figure 2: Tenure & Monthly Charges vs Churn', fontsize=12)
plt.tight_layout()
plt.savefig('chart_02_tenure_charges.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA Finding 2: Customers with tenure <12 months and higher monthly charges churn significantly more.')

EDA Finding 2: Customers with tenure <12 months and higher monthly charges churn significantly more.


In [10]:
# ── EDA Chart 3: Churn by Contract & Internet Service ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby(['Contract','Churn']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[0], color=['#4CAF50','#F44336'], edgecolor='black', rot=15)
axes[0].set_title('Churn by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Contract Type'); axes[0].set_ylabel('Count')
axes[0].legend(['No Churn','Churn'])

df.groupby(['InternetService','Churn']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[1], color=['#4CAF50','#F44336'], edgecolor='black', rot=15)
axes[1].set_title('Churn by Internet Service', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Internet Service'); axes[1].set_ylabel('Count')
axes[1].legend(['No Churn','Churn'])

plt.suptitle('Figure 3: Churn by Contract & Internet Service', fontsize=12)
plt.tight_layout()
plt.savefig('chart_03_contract_internet.png', dpi=150, bbox_inches='tight')
plt.show()

In [11]:
# ── EDA Chart 4: Correlation Heatmap ─────────────────────────────────────────
df_tmp = df.copy()
for col in df_tmp.select_dtypes(include='object').columns:
    df_tmp[col] = LabelEncoder().fit_transform(df_tmp[col])

plt.figure(figsize=(14, 8))
corr = df_tmp.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Figure 4: Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Round 2 – Feature Engineering & Machine Learning

### Feature Engineering

In [12]:
# ── Feature Engineering ──────────────────────────────────────────────────────
df_model = df.copy()
addons = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport',
          'StreamingTV','StreamingMovies']
df_model['ChargesPerMonth']      = df_model['TotalCharges'] / (df_model['tenure'] + 1)
df_model['IsNewCustomer']        = (df_model['tenure'] <= 6).astype(int)
df_model['HasMultipleServices']  = (
    (df_model['PhoneService'] == 'Yes').astype(int) +
    (df_model['InternetService'] != 'No').astype(int)
)
df_model['NumAddOns'] = sum((df_model[c] == 'Yes').astype(int) for c in addons)
df_model['AutoPay']   = df_model['PaymentMethod'].str.contains('automatic').astype(int)
print('Engineered features: ChargesPerMonth, IsNewCustomer, HasMultipleServices, NumAddOns, AutoPay')

Engineered features: ChargesPerMonth, IsNewCustomer, HasMultipleServices, NumAddOns, AutoPay


In [13]:
# ── Encode Categorical Variables ─────────────────────────────────────────────
df_model.drop(columns=['customerID'], inplace=True, errors='ignore')

le = LabelEncoder()
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

multi_cat = list(df_model.select_dtypes(include='object').columns)
df_model = pd.get_dummies(df_model, columns=multi_cat, drop_first=True).astype(float)

print(f'NaN values after encoding: {df_model.isnull().sum().sum()}  (should be 0)')
print(f'Final shape after encoding: {df_model.shape}')

NaN values after encoding: 0  (should be 0)
Final shape after encoding: (7043, 36)


In [14]:
# ── Train-Test Split ─────────────────────────────────────────────────────────
# Imbalance is handled by threshold tuning (next cell), not SMOTE: SMOTE on one-hot
# columns creates impossible synthetic customers and skews the probabilities.
# Scaling lives inside each model Pipeline, so it is fit on training folds only.
X = df_model.drop('Churn', axis=1)
y = df_model['Churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')
print(f'Churn rate  train: {y_train.mean():.3f}  test: {y_test.mean():.3f}')

Train: 5634  |  Test: 1409
Churn rate  train: 0.265  test: 0.265


### ML Model Training & Evaluation

In [15]:
# ── Model Comparison: tuned with 5-fold CV, threshold tuned on out-of-fold data ─
# Everything is chosen on the TRAIN set only; the test set is scored once at the end.
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
candidates = {
    'Logistic Regression': (LogisticRegression(max_iter=2000),
        {'clf__C': np.logspace(-2, 1, 8)}),
    'Random Forest': (RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        {'clf__max_depth': [4, 6, 8, 12], 'clf__min_samples_leaf': [5, 10, 20],
         'clf__max_features': ['sqrt', 0.5]}),
    'XGBoost': (XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0, n_jobs=-1),
        {'clf__n_estimators': [100, 200, 300], 'clf__max_depth': [2, 3, 4],
         'clf__learning_rate': [0.02, 0.05, 0.1], 'clf__subsample': [0.7, 0.9, 1.0],
         'clf__colsample_bytree': [0.6, 0.8, 1.0]}),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=SEED),
        {'clf__n_estimators': [100, 200], 'clf__max_depth': [2, 3],
         'clf__learning_rate': [0.03, 0.05, 0.1], 'clf__subsample': [0.7, 0.9, 1.0]}),
}
grid_t = np.arange(0.20, 0.61, 0.01)

results = {}
print(f'{"Model":<20} {"CV-AUC":>8} {"Thr":>6} {"Accuracy":>9} {"F1":>8} {"ROC-AUC":>8}')
print('-' * 64)
for name, (est, grid) in candidates.items():
    pipe = Pipeline([('sc', StandardScaler()), ('clf', est)])
    search = RandomizedSearchCV(pipe, grid, n_iter=8, cv=cv, scoring='roc_auc',
                                random_state=SEED, n_jobs=-1).fit(X_train, y_train)
    model = search.best_estimator_

    # Decision threshold: maximise mean(accuracy, F1) on out-of-fold train predictions
    oof = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    obj = [(accuracy_score(y_train, oof >= t) + f1_score(y_train, oof >= t)) / 2 for t in grid_t]
    thr = grid_t[int(np.argmax(obj))]

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= thr).astype(int)
    r = {'Accuracy': accuracy_score(y_test, y_pred), 'F1-Score': f1_score(y_test, y_pred),
         'ROC-AUC': roc_auc_score(y_test, y_prob), 'model': model, 'y_pred': y_pred,
         'y_prob': y_prob, 'Threshold': thr, 'CV-AUC': search.best_score_, 'CV-Score': max(obj)}
    results[name] = r
    print(f'{name:<20} {r["CV-AUC"]:>8.4f} {thr:>6.2f} {r["Accuracy"]:>9.4f} '
          f'{r["F1-Score"]:>8.4f} {r["ROC-AUC"]:>8.4f}')

# Winner picked from train-only CV results (not from the test set)
best_name = max(results, key=lambda k: results[k]['CV-Score'])
THRESHOLD = results[best_name]['Threshold']
print(f'\nBest Model: {best_name}  (threshold={THRESHOLD:.2f}, '
      f'test F1={results[best_name]["F1-Score"]:.4f})')

Model                  CV-AUC    Thr  Accuracy       F1  ROC-AUC
----------------------------------------------------------------


Logistic Regression    0.8488   0.37    0.7786   0.6259   0.8457


Random Forest          0.8456   0.35    0.7771   0.6244   0.8436


XGBoost                0.8495   0.39    0.7842   0.6209   0.8455


Gradient Boosting      0.8495   0.39    0.7913   0.6231   0.8459

Best Model: Logistic Regression  (threshold=0.37, test F1=0.6259)


In [16]:
# ── Best Model Report ─────────────────────────────────────────────────────────
best = results[best_name]
print(f'=== {best_name} – Classification Report ===')
print(classification_report(y_test, best['y_pred'], target_names=['No Churn','Churn']))

=== Logistic Regression – Classification Report ===
              precision    recall  f1-score   support

    No Churn       0.88      0.81      0.84      1035
       Churn       0.57      0.70      0.63       374

    accuracy                           0.78      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.80      0.78      0.79      1409



In [17]:
# ── Confusion Matrix & ROC Curve ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'],
            linewidths=1, linecolor='gray')
axes[0].set_title(f'Figure 5: Confusion Matrix – {best_name}', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={res['ROC-AUC']:.3f})")
axes[1].plot([0,1],[0,1],'k--', label='Random')
axes[1].set_title('Figure 6: ROC Curve Comparison', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right', fontsize=9); axes[1].grid(True)

plt.tight_layout()
plt.savefig('chart_05_confusion_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [18]:
# ── Feature Importance ────────────────────────────────────────────────────────
best_model = best['model']
final = best_model.named_steps['clf']
if hasattr(final, 'feature_importances_'):
    feat_imp = pd.Series(final.feature_importances_, index=X.columns)
    xlabel = 'Importance Score'
else:  # logistic regression: |standardised coefficient|
    feat_imp = pd.Series(np.abs(final.coef_[0]), index=X.columns)
    xlabel = '|Standardised Coefficient|'
top15 = feat_imp.nlargest(15)
plt.figure(figsize=(10, 6))
colors_fi = sns.color_palette('RdYlGn_r', n_colors=15)
plt.barh(top15.index[::-1], top15.values[::-1], color=colors_fi[::-1], edgecolor='black')
plt.title(f'Figure 7: Top 15 Feature Importances – {best_name}', fontsize=14, fontweight='bold')
plt.xlabel(xlabel)
plt.tight_layout()
plt.savefig('chart_06_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
top_features = list(feat_imp.nlargest(5).index)
print('Top 5 features:')
for feat, imp in feat_imp.nlargest(5).items():
    print(f'  {feat}: {imp:.4f}')

Top 5 features:
  MonthlyCharges: 1.6216
  InternetService_Fiber optic: 1.3611
  tenure: 0.7630
  ChargesPerMonth: 0.7580
  Contract_Two year: 0.6938


## Round 3 – Intelligence Layer, Visualization & Prototype

In [19]:
# ── Model Comparison Dashboard ───────────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {'Accuracy':v['Accuracy'],'F1-Score':v['F1-Score'],'ROC-AUC':v['ROC-AUC']}
    for name, v in results.items()}).T

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(metrics_df)); w = 0.25
b1 = ax.bar(x-w, metrics_df['Accuracy'], w, label='Accuracy',  color='#2196F3', edgecolor='black')
b2 = ax.bar(x,   metrics_df['F1-Score'], w, label='F1-Score',  color='#FF9800', edgecolor='black')
b3 = ax.bar(x+w, metrics_df['ROC-AUC'],  w, label='ROC-AUC',   color='#9C27B0', edgecolor='black')
for bs in [b1,b2,b3]:
    for b in bs:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metrics_df.index, rotation=10)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Figure 8: Model Comparison Dashboard', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('chart_07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [20]:
# ── Churn Risk Segmentation ───────────────────────────────────────────────────
# Low = below the model's decision threshold (predicted to stay); High >= 0.70.
RISK_BINS   = [0, THRESHOLD, 0.7, 1.0]
RISK_LABELS = ['Low Risk', 'Medium Risk', 'High Risk']
def risk_level(p):
    return RISK_LABELS[0] if p < RISK_BINS[1] else (RISK_LABELS[1] if p < RISK_BINS[2] else RISK_LABELS[2])

X_test_df = X_test.copy()
X_test_df['ChurnProbability'] = best['y_prob']
X_test_df['RiskSegment'] = pd.cut(X_test_df['ChurnProbability'], bins=RISK_BINS,
                                  labels=RISK_LABELS, include_lowest=True)

seg_counts = X_test_df['RiskSegment'].value_counts().reindex(RISK_LABELS)
actual_rate = y_test.groupby(X_test_df['RiskSegment'], observed=False).mean().reindex(RISK_LABELS)
plt.figure(figsize=(8,5))
bars = plt.bar(seg_counts.index, seg_counts.values,
               color=['#4CAF50','#FF9800','#F44336'], edgecolor='black', width=0.5)
for bar, val, rate in zip(bars, seg_counts.values, actual_rate.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
             f'{val}\n(actual churn {rate:.0%})', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, seg_counts.max()*1.2)
plt.title('Figure 9: Churn Risk Segmentation', fontsize=14, fontweight='bold')
plt.xlabel('Risk Segment'); plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('chart_08_risk_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Risk Segment Breakdown (count, observed churn rate):')
print(pd.DataFrame({'Customers': seg_counts, 'ActualChurnRate': actual_rate.round(3)}).to_string())

Risk Segment Breakdown (count, observed churn rate):
             Customers  ActualChurnRate
RiskSegment                            
Low Risk           949            0.119
Medium Risk        345            0.490
High Risk          115            0.800


In [21]:
# ── Prototype: Churn Risk Predictor ──────────────────────────────────────────
# Unspecified fields default to the training-set median/mode (a typical customer),
# not to 0, which would describe an impossible customer.
BASELINE = X_train.median().to_dict()

def predict_churn_risk(tenure, monthly_charges, contract='Month-to-month',
                       internet='Fiber optic', total_charges=None):
    row = dict(BASELINE)
    total_charges = tenure * monthly_charges if total_charges is None else total_charges
    row.update({'tenure': tenure, 'MonthlyCharges': monthly_charges,
                'TotalCharges': total_charges,
                'ChargesPerMonth': total_charges / (tenure + 1),
                'IsNewCustomer': int(tenure <= 6),
                'Contract_One year': int(contract == 'One year'),
                'Contract_Two year': int(contract == 'Two year'),
                'InternetService_Fiber optic': int(internet == 'Fiber optic')})
    prob = best_model.predict_proba(pd.DataFrame([row], columns=X.columns))[0, 1]
    return {'Churn Probability': f'{prob:.1%}', 'Risk Level': risk_level(prob).upper()}

print('=== CHURN PREDICTOR PROTOTYPE ===')
print('Case 1 (New customer, high charges, month-to-month, fibre):')
print(predict_churn_risk(tenure=2, monthly_charges=85, contract='Month-to-month'))
print()
print('Case 2 (Long-term, stable charges, two-year contract, DSL):')
print(predict_churn_risk(tenure=48, monthly_charges=45, contract='Two year', internet='DSL'))

=== CHURN PREDICTOR PROTOTYPE ===
Case 1 (New customer, high charges, month-to-month, fibre):
{'Churn Probability': '52.8%', 'Risk Level': 'MEDIUM RISK'}

Case 2 (Long-term, stable charges, two-year contract, DSL):
{'Churn Probability': '3.5%', 'Risk Level': 'LOW RISK'}


## Final Submission – Summary & Conclusion

In [22]:
# ── Final Results Summary ─────────────────────────────────────────────────────
best_res = results[best_name]
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean())
m2m, two = churn_by_contract['Month-to-month'], churn_by_contract['Two year']
long_tenure_churn = (df.loc[df['tenure'] > 24, 'Churn'] == 'Yes').mean()
early_churn       = (df.loc[df['tenure'] <= 6,  'Churn'] == 'Yes').mean()
print('=' * 60)
print('         DS SPRINT ASSESSMENT – FINAL RESULTS')
print('=' * 60)
print(f'Challenge  : Telco Customer Churn Prediction')
print(f'Dataset    : IBM Telco ({df.shape[0]} records, {df.shape[1]} features)')
print(f'Best Model : {best_name}  (decision threshold {THRESHOLD:.2f})')
print(f'Accuracy   : {best_res["Accuracy"]*100:.2f}%')
print(f'F1-Score   : {best_res["F1-Score"]:.4f}')
print(f'ROC-AUC    : {best_res["ROC-AUC"]:.4f}')
print('=' * 60)
print()
print('KEY INSIGHTS:')
print(f'  1. Month-to-month contracts churn {m2m:.1%} vs {two:.1%} for two-year ({m2m/two:.0f}x).')
print(f'  2. Customers with tenure <=6 months churn {early_churn:.1%}; tenure >24 months only {long_tenure_churn:.1%}.')
print(f'  3. Top model features: {", ".join(top_features)}.')
print()
print('RECOMMENDATIONS:')
print('  1. Offer loyalty discounts in the first 6 months to reduce early-stage churn.')
print('  2. Incentivise annual contract upgrades with exclusive perks for high-risk customers.')
print()
print('CONCLUSION:')
print(f'  {best_name} predicts churn with {best_res["Accuracy"]*100:.1f}% accuracy,')
print(f'  {best_res["F1-Score"]:.2f} F1 and {best_res["ROC-AUC"]:.2f} ROC-AUC on a held-out test set.')
print('  Hyper-parameters (5-fold CV) and the decision threshold were tuned on training data only.')
print('=' * 60)

         DS SPRINT ASSESSMENT – FINAL RESULTS
Challenge  : Telco Customer Churn Prediction
Dataset    : IBM Telco (7043 records, 21 features)
Best Model : Logistic Regression  (decision threshold 0.37)
Accuracy   : 77.86%
F1-Score   : 0.6259
ROC-AUC    : 0.8457

KEY INSIGHTS:
  1. Month-to-month contracts churn 42.7% vs 2.8% for two-year (15x).
  2. Customers with tenure <=6 months churn 52.9%; tenure >24 months only 14.0%.
  3. Top model features: MonthlyCharges, InternetService_Fiber optic, tenure, ChargesPerMonth, Contract_Two year.

RECOMMENDATIONS:
  1. Offer loyalty discounts in the first 6 months to reduce early-stage churn.
  2. Incentivise annual contract upgrades with exclusive perks for high-risk customers.

CONCLUSION:
  Logistic Regression predicts churn with 77.9% accuracy,
  0.63 F1 and 0.85 ROC-AUC on a held-out test set.
  Hyper-parameters (5-fold CV) and the decision threshold were tuned on training data only.


In [23]:
# ── Export Excel Assessment Sheet ────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

best_res = results[best_name]
dupes_count = 0  # already removed earlier

wb = Workbook()
ws = wb.active
ws.title = 'DS Sprint Assessment'

thin    = Side(style='thin', color='9E9E9E')
border  = Border(left=thin, right=thin, top=thin, bottom=thin)
center  = Alignment(horizontal='center', vertical='center', wrap_text=True)
left    = Alignment(horizontal='left',   vertical='center', wrap_text=True)

def set_cell(ws, row, col, value, bold=False, bg=None, fg='000000'):
    c = ws.cell(row=row, column=col, value=value)
    c.border  = border
    c.alignment = left
    c.font    = Font(bold=bold, color=fg)
    if bg: c.fill = PatternFill('solid', fgColor=bg)
    return c

def section_row(ws, row, title):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=3)
    c = ws.cell(row=row, column=1, value=title)
    c.font      = Font(bold=True, color='FFFFFF', size=11)
    c.fill      = PatternFill('solid', fgColor='1A237E')
    c.alignment = center
    c.border    = border

def data_row(ws, row, label, value):
    set_cell(ws, row, 1, label, bold=True, bg='E8EAF6')
    ws.merge_cells(start_row=row, start_column=2, end_row=row, end_column=3)
    set_cell(ws, row, 2, value, bg='FFFFFF')
    ws.row_dimensions[row].height = 35

# Title
ws.merge_cells('A1:C1')
c = ws['A1']
c.value = 'DS SPRINT ASSESSMENT – TELCO CUSTOMER CHURN PREDICTION'
c.font  = Font(bold=True, size=14, color='FFFFFF')
c.fill  = PatternFill('solid', fgColor='0D1B2A')
c.alignment = center
c.border = border
ws.row_dimensions[1].height = 32

row = 2

# Student details
section_row(ws, row, 'STUDENT DETAILS'); row += 1
for lbl, val in [
    ('Name', '(Your Full Name)'),
    ('Register Number', '(Your Register Number)'),
    ('Class', '2024 CSE'),
    ('Challenge ID', 'DS-CHN-01'),
    ('Challenge Title', 'Telco Customer Churn Prediction'),
]:
    data_row(ws, row, lbl, val); row += 1

# Round 1
row += 1
section_row(ws, row, 'ROUND 1 – DATA UNDERSTANDING, PREPARATION & EDA'); row += 1
for lbl, val in [
    ('Problem Statement',
     'Telecom companies lose revenue due to customer churn. This project '
     'predicts whether a customer will leave the service using demographic, '
     'account, and usage data to enable proactive retention strategies.'),
    ('Dataset Name', 'IBM Telco Customer Churn Dataset'),
    ('Dataset Source', 'IBM Sample Data / Kaggle (WA_Fn-UseC_-Telco-Customer-Churn.csv)'),
    ('Number of Records', str(df.shape[0])),
    ('Number of Attributes', str(df.shape[1])),
    ('Missing Value Handling',
     f'TotalCharges contained {missing_before} blank entries (all customers with tenure = 0, '
     'not yet billed). Converted to NaN and filled with 0.'),
    ('Duplicate Removal', '0 duplicate rows found. Dataset integrity confirmed.'),
    ('Outlier Treatment',
     'IQR check on tenure, MonthlyCharges and TotalCharges found no outliers; '
     'no values were clipped.'),
    ('EDA Finding 1',
     f'{churn_pct["Yes"]:.1f}% of customers churned (imbalanced dataset). '
     'Imbalance handled by tuning the decision threshold on out-of-fold predictions.'),
    ('EDA Finding 2',
     f'Customers with tenure <= 6 months churn {early_churn:.1%} vs {long_tenure_churn:.1%} '
     f'beyond 24 months. Month-to-month contracts churn {m2m:.1%} vs {two:.1%} for 2-year.'),
]:
    data_row(ws, row, lbl, val); row += 1

# Round 2
row += 1
section_row(ws, row, 'ROUND 2 – FEATURE ENGINEERING & MACHINE LEARNING'); row += 1
for lbl, val in [
    ('Selected Input Features',
     'tenure, MonthlyCharges, TotalCharges, Contract, InternetService, '
     'PaymentMethod, PaperlessBilling, TechSupport, OnlineSecurity, '
     'ChargesPerMonth [engineered], IsNewCustomer [engineered], '
     'HasMultipleServices [engineered], NumAddOns [engineered], AutoPay [engineered]'),
    ('Target Variable', 'Churn  (Binary: Yes = 1, No = 0)'),
    ('ML Algorithm Used', best_name),
    ('Justification',
     f'{best_name} had the best cross-validated accuracy/F1 trade-off among four tuned '
     f'candidates (LR, RF, XGBoost, GB). Decision threshold {THRESHOLD:.2f} tuned on '
     'out-of-fold training predictions; metrics below are on an untouched test set.'),
    ('Accuracy',  f"{best_res['Accuracy']*100:.2f}%"),
    ('F1-Score',  f"{best_res['F1-Score']:.4f}"),
    ('ROC-AUC',   f"{best_res['ROC-AUC']:.4f}"),
]:
    data_row(ws, row, lbl, val); row += 1

# Round 3
row += 1
section_row(ws, row, 'ROUND 3 – INTELLIGENCE LAYER, VISUALIZATION & PROTOTYPE'); row += 1
for lbl, val in [
    ('Visualizations Created',
     '1. Churn Distribution – Bar & Pie Chart  '
     '2. Tenure & Monthly Charges Histogram + Boxplot  '
     '3. Churn by Contract Type & Internet Service  '
     '4. Feature Correlation Heatmap  '
     '5. Confusion Matrix & ROC Curve  '
     '6. Top 15 Feature Importance Chart  '
     '7. Model Comparison Dashboard  '
     '8. Churn Risk Segmentation Chart'),
    ('Key Insight 1',
     f'Month-to-month contract customers churn {m2m:.1%}, vs {churn_by_contract["One year"]:.1%} '
     f'for one-year and {two:.1%} for two-year holders. '
     'Contract type is the most actionable business lever.'),
    ('Key Insight 2',
     f'Top model features: {", ".join(top_features)}. Customers beyond 24 months churn '
     f'at {long_tenure_churn:.1%}; early intervention (months 1–6) is critical.'),
    ('Prototype / Intelligence Layer',
     'A Python Churn Risk Predictor function was implemented that accepts '
     'customer attributes and returns churn probability with risk classification '
     '(Low / Medium / High). All test customers were risk-segmented.'),
    ('Recommendation 1',
     'Offer new customers (tenure ≤ 6 months) a 15% discount or free '
     'premium feature for 3 months to prevent early-stage churn.'),
    ('Recommendation 2',
     'Proactively contact Medium/High Risk customers (probability above the model threshold) with '
     'personalised retention offers — contract upgrade incentives, '
     'technical support bundles, or billing discounts.'),
]:
    data_row(ws, row, lbl, val); row += 1

# Final submission
row += 1
section_row(ws, row, 'FINAL SUBMISSION'); row += 1
for lbl, val in [
    ('GitHub Repository Link', '(Paste your GitHub link here)'),
    ('Overall Result / Conclusion',
     f'The {best_name} model predicts customer churn with '
     f'{best_res["Accuracy"]*100:.1f}% accuracy and {best_res["F1-Score"]:.2f} '
     f'F1-score (ROC-AUC: {best_res["ROC-AUC"]:.2f}) on a held-out test set. '
     'CV-tuned models, threshold tuning and engineered features drive the result. '
     'The intelligence layer prototype enables real-time '
     'churn risk scoring, directly supporting data-driven customer retention decisions.'),
]:
    data_row(ws, row, lbl, val); row += 1

ws.column_dimensions['A'].width = 28
ws.column_dimensions['B'].width = 60
ws.column_dimensions['C'].width = 18

wb.save('DS_Sprint_Assessment_Sheet.xlsx')
print('Excel sheet saved: DS_Sprint_Assessment_Sheet.xlsx')

Excel sheet saved: DS_Sprint_Assessment_Sheet.xlsx


In [24]:
# ── List all output files ─────────────────────────────────────────────────────
import os
print('Generated output files:')
files = [f for f in os.listdir('.') if f.endswith(('.png', '.xlsx', '.csv'))]
for f in sorted(files):
    size = os.path.getsize(f)
    print(f'  {f:<50} {size/1024:>8.1f} KB')

Generated output files:
  DS_Sprint_Assessment_Sheet.xlsx                         7.6 KB
  chart_01_churn_distribution.png                        67.7 KB
  chart_02_tenure_charges.png                            69.4 KB
  chart_03_contract_internet.png                         69.1 KB
  chart_04_correlation_heatmap.png                      356.4 KB
  chart_05_confusion_roc.png                            114.9 KB
  chart_06_feature_importance.png                        79.6 KB
  chart_07_model_comparison.png                          57.4 KB
  chart_08_risk_segmentation.png                         46.3 KB
  telco_churn.csv                                       947.7 KB
